# BT4012 Competition 2026 – Illicit Bitcoin Transaction Detection
## Final model: reproduces the two selected submissions

| Selected submission | Public AUC | What it is |
|---|---|---|
| `submission_final_top3.csv` | **0.95585** | Rank average of E1, E2 and E3 (below) |
| `submission_v10_E1.csv` | **0.95582** | Window bag: LightGBM trained on the last 16/18/20/22/24 time steps, 10 seeds each, rank-averaged |

**Pipeline overview**
1. **Load data** (section 2). Labels: `1` = illicit, `0` = licit, `none` = unlabelled.
2. **Diagnosis** (section 3). `test.csv` contains only the *labelled* transactions of steps 36–49; the unlabelled transactions of that period were removed.
3. **Graph features on a labelled-only graph** (section 4). Degrees, 2-hop path counts, component size, and the average local features of a transaction's predecessors and successors, computed over labelled-training + test transactions only. This way train and test transactions see the same kind of neighbourhood (fixing a train/test mismatch that cost 0.012 AUC in our first version).
4. **Drift check** (section 5): adversarial validation showing the mismatch and the fix.
5. **Time-based validation** (section 6): forward-chaining folds (train on earlier steps, validate on 5-step windows 21–25, 26–30, 31–35), one tree count shared by all folds.
6. **External data** (sections 9–10): the public Elliptic *features* file (no labels; permitted by the instructor) restores the removed test-period transactions, so graph features can also be computed on the *complete* graph. Used by one ingredient (E2).
7. **Final models** (section 11): LightGBM window bag (E1), plus two variants for the blend (E2, E3), then the top-3 rank average.

Sections 6b and 7 compute extra features explored during the competition; they are kept for completeness but are **not used** by the final models.

**How to run:** put `train.csv`, `test.csv`, `txs_edgelist.csv`, `sample_submission.csv` in `data/` (next to the notebook or one folder up) and the public Elliptic `elliptic_txs_features.csv` anywhere in the project folder (e.g. `data/elliptic_bitcoin_dataset/`). Install the packages in `requirements.txt`, then *Run All* (≈ 30–40 min on a laptop). Outputs: `reproduced_submission_final_top3.csv` and `reproduced_submission_v10_E1.csv`, plus a check against the originally submitted files if they are in the folder.

In [1]:
# ===== 1. Setup =====
import os, glob, random, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_data_dir():
    # Look for the competition files locally first, then in Kaggle's input folder
    candidates = ['./data', '.', '../data', '..'] + glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
    for d in candidates:
        if os.path.exists(os.path.join(d, 'train.csv')):
            return d
    raise FileNotFoundError('Put train.csv, test.csv, txs_edgelist.csv and sample_submission.csv in ./data')

DATA_DIR = find_data_dir()
OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
REPORT = {}   # key numbers collected along the way, saved at the end for the report
print('Data dir:', DATA_DIR, '| Output dir:', OUT_DIR)

Data dir: ./data | Output dir: .


## 2. Load data
Unchanged from v1. Feature columns are detected automatically. The first 93 are **local** features (the transaction itself) and the other 72 are **aggregated** features (statistics over 1-hop neighbours, pre-computed by the dataset authors on the *full* graph, so they are consistent between train and test). `time_step` is not used as a feature because every test step is unseen.

In [2]:
# ===== 2. Load data =====
t0 = time.time()
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
edges = pd.read_csv(os.path.join(DATA_DIR, 'txs_edgelist.csv'))
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
print(f'train {train.shape} | test {test.shape} | edges {edges.shape} | sample {sample.shape} | {time.time() - t0:.0f}s')

NON_FEAT = {'txid', 'time_step', 'timestep', 'time step', 'label', 'class', 'y', 'index', 'is_test'}
FEAT = [c for c in train.columns
        if c.strip().lower() not in NON_FEAT and c in test.columns
        and pd.api.types.is_numeric_dtype(train[c])]
local_named = [c for c in FEAT if 'local' in c.lower()]
LOCAL = local_named if local_named else FEAT[:93]
AGG = [c for c in FEAT if c not in LOCAL]
print(len(FEAT), 'features:', len(LOCAL), 'local +', len(AGG), 'aggregated')
assert len(FEAT) > 0 and len(LOCAL) > 0, 'No feature columns found - check train.columns'

for df in (train, test):
    df[FEAT] = df[FEAT].astype(np.float32)

# Label cleaning: 1 = illicit, 0 = licit, anything else -> NaN = unlabelled
lab = train['label'].astype(str).str.strip().str.lower()
train['y'] = lab.map({'1': 1, '1.0': 1, 'illicit': 1, '0': 0, '0.0': 0, 'licit': 0})
print(train['y'].value_counts(dropna=False))

train (141772, 168) | test (15329, 168) | edges (234355, 2) | sample (15329, 2) | 2s
165 features: 93 local + 72 aggregated
y
NaN    110537
0.0     27591
1.0      3644
Name: count, dtype: int64


## 3. Diagnosis: what is in the test set?
The training file contains every transaction in steps 1–35 (labelled and unlabelled). The test file only contains **labelled** transactions from steps 36–49. The cell below checks this by counting edges whose endpoints are missing from both files: those edges point to the removed unlabelled test-period transactions.

In [3]:
# ===== 3. Diagnosis =====
train_ids, test_ids = set(train['txId']), set(test['txId'])
e1, e2 = edges.iloc[:, 0], edges.iloc[:, 1]
in_train = e1.isin(train_ids) & e2.isin(train_ids)
in_test = e1.isin(test_ids) & e2.isin(test_ids)
missing = ~(e1.isin(train_ids | test_ids) & e2.isin(train_ids | test_ids))
print(f'Edges inside train: {in_train.sum():,} | inside test: {in_test.sum():,} | '
      f'touching a transaction missing from both files: {missing.sum():,}')
print(f'Labelled share of train rows: {train["y"].notna().mean():.1%} | test rows: assumed 100% labelled')
REPORT['edges_missing_endpoint'] = int(missing.sum())
REPORT['train_labelled_share'] = float(train['y'].notna().mean())

Edges inside train: 163,194 | inside test: 12,724 | touching a transaction missing from both files: 58,437
Labelled share of train rows: 22.0% | test rows: assumed 100% labelled


## 4. Graph feature engineering (fixed)
Same features as v1: degrees, 2-hop reach, neighbours' average degree, component size, direction-aware neighbour means of the 93 local features, and per-step percentile ranks. **The only change is which transactions form the graph.**

- **v1 graph:** all training transactions + test transactions. Training transactions keep their unlabelled neighbours, but test transactions have lost theirs.
- **v2 graph:** labelled training transactions + test transactions. Every transaction, in train or test, only sees neighbours that are labelled. Train and test features now mean the same thing.

Edges never cross time steps in this dataset, so dropping unlabelled training transactions does not touch the test graph at all.

The table printed below compares the average degree of training vs test transactions under both graphs. Under v1 the two should be clearly different; under v2 they should be much closer.

In [4]:
# ===== 4. Graph features =====
train['is_test'] = 0
test['is_test'] = 1

def mean_over_neighbours(M, X):
    # Average of X over each node's neighbours in M (0 when the node has no neighbours)
    deg = np.asarray(M.sum(axis=1)).ravel()
    inv = np.zeros(len(deg), dtype=np.float32)
    inv[deg > 0] = 1.0 / deg[deg > 0]
    return np.asarray(sp.diags(inv) @ (M @ X), dtype=np.float32)

def graph_features(nodes, with_neighbour_means=True):
    """Graph features for every row of `nodes`, using only edges between rows of `nodes`."""
    N = len(nodes)
    row_of = pd.Series(np.arange(N), index=nodes['txId'].values)
    src, dst = edges.iloc[:, 0].map(row_of), edges.iloc[:, 1].map(row_of)
    keep = (src.notna() & dst.notna()).values
    src, dst = src[keep].astype(np.int64).values, dst[keep].astype(np.int64).values
    A = sp.csr_matrix((np.ones(len(src), dtype=np.float32), (src, dst)), shape=(N, N))
    A.data[:] = 1.0                       # collapse duplicate edges
    AT = A.T.tocsr()                      # AT[i, j] = 1 : j is a predecessor of i
    in_deg = np.asarray(AT.sum(axis=1)).ravel()
    out_deg = np.asarray(A.sum(axis=1)).ravel()
    _, comp = connected_components(A, directed=True, connection='weak')
    g = pd.DataFrame({
        'g_in_deg': in_deg, 'g_out_deg': out_deg, 'g_deg': in_deg + out_deg,
        'g_in_out_ratio': (in_deg + 1) / (out_deg + 1),
        'g_out_2hop': A @ out_deg, 'g_in_2hop': AT @ in_deg,
        'g_pred_mean_out_deg': mean_over_neighbours(AT, out_deg.reshape(-1, 1)).ravel(),
        'g_succ_mean_in_deg': mean_over_neighbours(A, in_deg.reshape(-1, 1)).ravel(),
        'g_comp_size': np.bincount(comp)[comp],
    }, index=nodes.index).astype(np.float32)
    if not with_neighbour_means:
        return g, int(keep.sum())
    X = nodes[LOCAL].values.astype(np.float32)
    pred = pd.DataFrame(mean_over_neighbours(AT, X), columns=['pred_mean_' + c for c in LOCAL], index=nodes.index)
    succ = pd.DataFrame(mean_over_neighbours(A, X), columns=['succ_mean_' + c for c in LOCAL], index=nodes.index)
    return pd.concat([g, pred, succ], axis=1), int(keep.sum())

all_nodes = pd.concat([train, test], ignore_index=True, sort=False)
assert all_nodes['txId'].is_unique, 'duplicate txIds between train and test'

# v1 graph (all transactions) - structural features only, kept just for the comparison
g_v1, n_e1 = graph_features(all_nodes, with_neighbour_means=False)
# v2 graph (labelled training + test transactions) - used for modelling
nodes = all_nodes[(all_nodes['is_test'] == 1) | all_nodes['y'].notna()].reset_index(drop=True)
g_v2, n_e2 = graph_features(nodes)
print(f'Edges used - v1 graph: {n_e1:,} | v2 graph: {n_e2:,}')

# Per-time-step percentile ranks, among labelled + test transactions only
RANK = ['rank_' + c for c in LOCAL]
rank_df = nodes.groupby('time_step')[LOCAL].rank(pct=True).astype(np.float32)
rank_df.columns = RANK

GSTRUCT = [c for c in g_v2.columns if c.startswith('g_')]
GRAPH = list(g_v2.columns)
nodes = pd.concat([nodes, g_v2, rank_df], axis=1)

L = nodes[nodes['is_test'] == 0].reset_index(drop=True)   # labelled training rows
T = nodes[nodes['is_test'] == 1].reset_index(drop=True)   # test rows, original order
y = L['y'].astype(int).values
print(f'Labelled train rows: {len(L):,} ({y.mean():.1%} illicit) | test rows: {len(T):,}')

# Evidence for the report: average degree of labelled-train vs test transactions under each graph
lab_mask = (all_nodes['is_test'] == 0) & all_nodes['y'].notna()
tst_mask = all_nodes['is_test'] == 1
cmp = pd.DataFrame({
    'v1 train (labelled)': g_v1[lab_mask.values][GSTRUCT].mean(),
    'v1 test': g_v1[tst_mask.values][GSTRUCT].mean(),
    'v2 train (labelled)': L[GSTRUCT].mean(),
    'v2 test': T[GSTRUCT].mean(),
}).round(2)
display(cmp)
for c in ['g_deg', 'g_comp_size']:
    REPORT[f'{c}_v1_train'], REPORT[f'{c}_v1_test'] = cmp.loc[c, 'v1 train (labelled)'], cmp.loc[c, 'v1 test']
    REPORT[f'{c}_v2_train'], REPORT[f'{c}_v2_test'] = cmp.loc[c, 'v2 train (labelled)'], cmp.loc[c, 'v2 test']

Edges used - v1 graph: 175,918 | v2 graph: 36,624
Labelled train rows: 31,235 (11.7% illicit) | test rows: 15,329


,v1 train (labelled),v1 test,v2 train (labelled),v2 test
g_in_deg,1.760000,0.830000,0.770000,0.830000
g_out_deg,1.100000,0.830000,0.770000,0.830000
g_deg,2.870000,1.660000,1.530000,1.660000
g_in_out_ratio,1.860000,1.230000,1.250000,1.230000
g_out_2hop,0.930000,0.730000,0.520000,0.730000
g_in_2hop,1.520000,0.730000,0.520000,0.730000
g_pred_mean_out_deg,10.610000,1.350000,9.410000,1.350000
g_succ_mean_in_deg,8.850000,6.370000,2.440000,6.370000
g_comp_size,4780.149902,432.459991,297.799988,432.459991


## 5. Drift check on the graph features: v1 vs v2 (kept from v2 for the report)
**Adversarial validation:** a classifier is trained to tell labelled training transactions apart from test transactions, using **only the 9 structural graph features**. An AUC near 0.5 means it cannot tell them apart (no drift). An AUC near 1.0 means the features look completely different in test. If the fix works, the v2 AUC should be clearly lower than v1.

Some drift will remain in both, because the test period is genuinely different (later time steps, and a known change in illicit behaviour after a major dark market shut down around step 43).

In [5]:
# ===== 5. Adversarial validation: v1 vs v2 graph features =====
def adversarial_auc(X_train_rows, X_test_rows, return_model=False):
    X = pd.concat([X_train_rows, X_test_rows], ignore_index=True)
    a = np.r_[np.zeros(len(X_train_rows)), np.ones(len(X_test_rows))]
    p = np.zeros(len(X))
    for tr, va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, a):
        m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31,
                               colsample_bytree=0.5, random_state=SEED, verbose=-1)
        m.fit(X.iloc[tr], a[tr])
        p[va] = m.predict_proba(X.iloc[va])[:, 1]
    auc = roc_auc_score(a, p)
    if return_model:
        m.fit(X, a)
        return auc, pd.Series(m.booster_.feature_importance('gain'), index=X.columns).sort_values(ascending=False)
    return auc

adv_v1 = adversarial_auc(g_v1[lab_mask.values][GSTRUCT], g_v1[tst_mask.values][GSTRUCT])
adv_v2 = adversarial_auc(L[GSTRUCT], T[GSTRUCT])
print(f'Train-vs-test AUC on graph features  v1: {adv_v1:.3f}   v2: {adv_v2:.3f}   (0.5 = no drift)')
REPORT['adv_auc_graph_v1'], REPORT['adv_auc_graph_v2'] = adv_v1, adv_v2
del g_v1, all_nodes

Train-vs-test AUC on graph features  v1: 1.000   v2: 0.931   (0.5 = no drift)


## 6. Time-based validation (stricter)
Forward-chaining folds as before: train on all earlier steps, validate on a later window. Changes:
- **Windows are 5 steps wide** (21–25, 26–30, 31–35) instead of 3. The test set runs 1–14 steps after training, so validating further ahead is more realistic.
- **One tree count for all folds.** Each fold records its AUC after every tree; the curves are averaged, and the peak of the average curve gives a single tree count. v1 let each fold stop at its own best point, which flatters the score.
- Because the graph is now built the same way as for test, validation transactions have exactly the same kind of features the test transactions have.

In [6]:
# ===== 6. Folds and model helpers =====
steps = np.sort(L['time_step'].unique())
N_FOLDS, VAL_WIDTH = 3, 5
folds = []
for k in range(N_FOLDS, 0, -1):
    end = len(steps) - (k - 1) * VAL_WIDTH
    val_steps = steps[end - VAL_WIDTH:end]
    tr_idx = np.where(L['time_step'] < val_steps.min())[0]
    va_idx = np.where(L['time_step'].isin(val_steps))[0]
    if len(np.unique(y[va_idx])) < 2:
        continue
    folds.append((tr_idx, va_idx))
    print(f'Fold {len(folds)}: train steps {steps.min()}-{val_steps.min() - 1} ({len(tr_idx):,} rows) | '
          f'validate steps {val_steps.min()}-{val_steps.max()} ({len(va_idx):,} rows, {y[va_idx].mean():.1%} illicit)')

BASE_PARAMS = dict(objective='binary', metric='auc', learning_rate=0.04, num_leaves=31,
                   min_child_samples=30, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                   reg_lambda=1.0, n_estimators=1500, n_jobs=-1, verbose=-1,
                   deterministic=True, force_row_wise=True)
REG_PARAMS = dict(BASE_PARAMS, num_leaves=15, min_child_samples=100, colsample_bytree=0.3,
                  reg_lambda=5.0, reg_alpha=1.0)
PARAMS = BASE_PARAMS   # may be replaced by REG_PARAMS in section 8

def recency_weights(t):
    # Linear weights from 0.5 (oldest step) to 1.0 (newest step) within the training data
    return 0.5 + 0.5 * (t - t.min()) / max(1, t.max() - t.min())

def fit_lgb(X_tr, y_tr, params, n_estimators=None, seed=SEED, weights=None, X_va=None, y_va=None):
    p = dict(params, random_state=seed)
    if n_estimators is not None:
        p['n_estimators'] = n_estimators
    m = lgb.LGBMClassifier(**p)
    if X_va is not None:
        m.fit(X_tr, y_tr, sample_weight=weights, eval_set=[(X_va, y_va)], eval_metric='auc')
    else:
        m.fit(X_tr, y_tr, sample_weight=weights)
    return m

def cv_lgb(cols, params=None, recency=False):
    params = params or PARAMS
    models, curves = [], []
    for tr, va in folds:
        w = recency_weights(L.loc[tr, 'time_step'].values) if recency else None
        m = fit_lgb(L.loc[tr, cols], y[tr], params, weights=w, X_va=L.loc[va, cols], y_va=y[va])
        models.append(m)
        curves.append(np.asarray(m.evals_result_['valid_0']['auc']))
    n = min(len(c) for c in curves)
    best = int(np.argmax(np.mean([c[:n] for c in curves], axis=0))) + 1   # one tree count for all folds
    preds = [m.predict_proba(L.loc[va, cols], num_iteration=best)[:, 1] for m, (_, va) in zip(models, folds)]
    aucs = [roc_auc_score(y[va], p) for p, (_, va) in zip(preds, folds)]
    return {'mean': float(np.mean(aucs)), 'aucs': aucs, 'iters': best, 'preds': preds}

results = {}
def run(name, cols, **kw):
    t0 = time.time()
    r = cv_lgb(cols, **kw)
    results[name] = r
    print(f'{name:42s} {len(cols):4d} feats | mean AUC {r["mean"]:.4f} | folds {np.round(r["aucs"], 4)} '
          f'| trees {r["iters"]} | {time.time() - t0:.0f}s')
    return r

Fold 1: train steps 1-20 (18,889 rows) | validate steps 21-25 (5,311 rows, 10.7% illicit)
Fold 2: train steps 1-25 (24,200 rows) | validate steps 26-30 (2,705 rows, 22.8% illicit)
Fold 3: train steps 1-30 (26,905 rows) | validate steps 31-35 (4,330 rows, 15.9% illicit)


## 6b. Extended graph features (new in v3)
Built on the same labelled + test graph as v2 (so they are consistent between train and test):

- **2-hop neighbour means** (`pred2_mean_*`, `succ2_mean_*`): the average of the local features over transactions two steps upstream / downstream. A 1-hop mean only sees direct neighbours; laundering patterns often run over several hops.
- **Neighbour spread** (`nb_std_*`): standard deviation of each local feature over all direct neighbours (both directions). Distinguishes a transaction surrounded by similar transactions from one whose neighbours are mixed.
- **Difference from neighbours** (`nb_diff_*`): the transaction's own value minus its neighbours' average. Captures "this transaction looks unusual compared with its surroundings". Set to 0 for transactions without neighbours.
- **2-hop counts** (`g2_n_pred2`, `g2_n_succ2`): how many distinct transactions are reachable in two steps.

In [7]:
# ===== 6b. Extended graph features =====
t0 = time.time()
N = len(nodes)
row_of = pd.Series(np.arange(N), index=nodes['txId'].values)
s_, d_ = edges.iloc[:, 0].map(row_of), edges.iloc[:, 1].map(row_of)
k_ = (s_.notna() & d_.notna()).values
s_, d_ = s_[k_].astype(np.int64).values, d_[k_].astype(np.int64).values
A = sp.csr_matrix((np.ones(len(s_), dtype=np.float32), (s_, d_)), shape=(N, N))
A.data[:] = 1.0
AT = A.T.tocsr()
U = ((A + AT) > 0).astype(np.float32).tocsr()          # undirected neighbours

def two_hop(M):
    M2 = (M @ M).tocsr()
    M2.data[:] = 1.0
    M2.setdiag(0)                                       # a transaction is not its own 2-hop neighbour
    M2.eliminate_zeros()
    return M2

A2, AT2 = two_hop(A), two_hop(AT)
X = nodes[LOCAL].values.astype(np.float32)
has_nb = np.asarray(U.sum(axis=1)).ravel() > 0
m_u = mean_over_neighbours(U, X)
std_u = np.sqrt(np.maximum(mean_over_neighbours(U, X ** 2) - m_u ** 2, 0)).astype(np.float32)
diff_u = np.where(has_nb[:, None], X - m_u, 0).astype(np.float32)

parts = [
    pd.DataFrame(mean_over_neighbours(AT2, X), columns=['pred2_mean_' + c for c in LOCAL]),
    pd.DataFrame(mean_over_neighbours(A2, X), columns=['succ2_mean_' + c for c in LOCAL]),
    pd.DataFrame(std_u, columns=['nb_std_' + c for c in LOCAL]),
    pd.DataFrame(diff_u, columns=['nb_diff_' + c for c in LOCAL]),
    pd.DataFrame({'g2_n_pred2': np.asarray(AT2.sum(axis=1)).ravel(),
                  'g2_n_succ2': np.asarray(A2.sum(axis=1)).ravel()}).astype(np.float32),
]
G2 = [c for p in parts for c in p.columns]
nodes = pd.concat([nodes] + [p.set_index(nodes.index) for p in parts], axis=1)
del X, m_u, std_u, diff_u, parts

L = nodes[nodes['is_test'] == 0].reset_index(drop=True)
T = nodes[nodes['is_test'] == 1].reset_index(drop=True)
assert (L['y'].astype(int).values == y).all()
print(f'{len(G2)} extended graph features added | {time.time() - t0:.0f}s')
print(f'Transactions with 2-hop predecessors: {(L["g2_n_pred2"] > 0).mean():.1%} (train) vs {(T["g2_n_pred2"] > 0).mean():.1%} (test)')
REPORT['n_extended_graph_features'] = len(G2)

374 extended graph features added | 0s
Transactions with 2-hop predecessors: 30.0% (train) vs 26.0% (test)


## 7. Idea 1 – structural features from the complete edge list
The graph is rebuilt from **every edge in `txs_edgelist.csv`**, including edges whose other end is an unlabelled transaction (present in `train.csv` for steps 1–35, removed from `test.csv` for steps 36–49 but still in the edge list). Only structure is used (counts), never features or labels of the missing transactions, so train and test are computed identically.

The table compares training vs test averages, and the drift check tells whether these features look the same in train and test (0.5 = identical).

In [8]:
# ===== 7. Full edge list structural features =====
t0 = time.time()
e_src, e_dst = edges.iloc[:, 0].values, edges.iloc[:, 1].values
ids_all = pd.unique(np.concatenate([e_src, e_dst, nodes['txId'].values]))
pos = pd.Series(np.arange(len(ids_all)), index=ids_all)
si, di = pos[e_src].values, pos[e_dst].values
M = len(ids_all)
F = sp.csr_matrix((np.ones(len(si), dtype=np.float32), (si, di)), shape=(M, M))
F.data[:] = 1.0
FT = F.T.tocsr()
present = np.zeros(M, dtype=np.float32)                 # 1 = labelled train or test transaction
present[pos[nodes['txId'].values].values] = 1.0
in_f, out_f = np.asarray(FT.sum(axis=1)).ravel(), np.asarray(F.sum(axis=1)).ravel()
in_lab, out_lab = FT @ present, F @ present
deg_f = in_f + out_f
full = pd.DataFrame({
    'f_in_deg': in_f, 'f_out_deg': out_f, 'f_deg': deg_f,
    'f_in_unlab': in_f - in_lab, 'f_out_unlab': out_f - out_lab,
    'f_lab_frac': np.where(deg_f > 0, (in_lab + out_lab) / np.maximum(deg_f, 1), -1),
    'f_out_2hop': F @ out_f, 'f_in_2hop': FT @ in_f,
    'f_pred_mean_out_deg': mean_over_neighbours(FT, out_f.reshape(-1, 1)).ravel(),
    'f_succ_mean_in_deg': mean_over_neighbours(F, in_f.reshape(-1, 1)).ravel(),
}).astype(np.float32)
FULLG = list(full.columns)
fn = full.iloc[pos[nodes['txId'].values].values].reset_index(drop=True)
fn.index = nodes.index
nodes = pd.concat([nodes, fn], axis=1)

nL = int((nodes['is_test'] == 0).sum())
L = nodes[nodes['is_test'] == 0].reset_index(drop=True)
T = nodes[nodes['is_test'] == 1].reset_index(drop=True)
assert (L['y'].astype(int).values == y).all()
assert (nodes['is_test'].values[:nL] == 0).all() and (nodes['is_test'].values[nL:] == 1).all()

display(pd.DataFrame({'train (labelled)': L[FULLG].mean(), 'test': T[FULLG].mean()}).round(2))
adv_full = adversarial_auc(L[FULLG], T[FULLG])
print(f'Train-vs-test drift AUC on full-edge-list features: {adv_full:.3f}  '
      f'(labelled-only graph features: {REPORT.get("adv_auc_graph_v2", float("nan")):.3f}; 0.5 = no drift) | {time.time() - t0:.0f}s')
REPORT['adv_auc_full_edge_features'] = adv_full

,train (labelled),test
f_in_deg,1.76,2.02
f_out_deg,1.10,1.22
f_deg,2.87,3.24
f_in_unlab,1.00,1.19
f_out_unlab,0.34,0.39
f_lab_frac,0.63,0.61
f_out_2hop,0.93,1.49
f_in_2hop,1.52,2.11
f_pred_mean_out_deg,10.61,7.10
f_succ_mean_in_deg,8.85,16.02


Train-vs-test drift AUC on full-edge-list features: 0.783  (labelled-only graph features: 0.931; 0.5 = no drift) | 3s


## 8. Helpers: recent-window folds, post-processing, submission writer
- `recent_folds(20)`: v4's setup, each fold trains only on the 20 time steps before its validation window.
- `nb_smooth`: idea 2. For each transaction, average the predictions of its neighbours (labelled-only graph from section 4, which on test contains exactly the test transactions) and mix with its own prediction.
- `step_rank`: idea 3. Percentile of each prediction within its time step.

In [9]:
# ===== 8. Helpers =====
TOL = 0.0005
SEEDS = [42, 7, 2026, 11, 123]
V4_PUBLIC = 0.95469
folds_full = list(folds)

def recent_folds(K):
    out = []
    for tr, va in folds_full:
        vmin = L.loc[va, 'time_step'].min()
        out.append((tr[L.loc[tr, 'time_step'].values >= vmin - K], va))
    return out

def rank01(p):
    return rankdata(p) / len(p)

def nb_smooth(node_idx, p, alpha):
    # node_idx: rows of `nodes` the predictions belong to; neighbours outside node_idx are ignored
    if alpha == 0:
        return p
    sub = U[node_idx][:, node_idx]
    cnt = np.asarray(sub.sum(axis=1)).ravel()
    nb = np.where(cnt > 0, (sub @ p) / np.maximum(cnt, 1), p)
    return (1 - alpha) * p + alpha * nb

def step_rank(steps, p):
    return pd.Series(p).groupby(np.asarray(steps)).rank(pct=True).values

def save_submission(pred, fname):
    if 'index' in test.columns:
        sub = sample[['index']].merge(pd.DataFrame({'index': test['index'].values, 'target': pred}),
                                      on='index', how='left')
    else:
        assert len(pred) == len(sample), 'test.csv and sample_submission.csv lengths differ'
        sub = pd.DataFrame({'index': sample['index'].values, 'target': pred})
    assert sub['target'].notna().all() and len(sub) == len(sample)
    path = os.path.join(OUT_DIR, fname)
    sub.to_csv(path, index=False)
    print('Saved', path, sub.shape)

def find(fname):
    for d in [OUT_DIR, '.', '..']:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

## 9. Load the public Elliptic features (external data, labels not used)
1. Find `elliptic_txs_features.csv` (no header: txId, time step, 165 features): searched in the data folder, the notebook folder, the two folders above it and ~/Downloads, including inside .zip files. If it is somewhere else, set `PUB_PATH_MANUAL` to its path.
2. Match it to the competition files by `txId`, and check that time steps agree and that features are identical or a simple rescaling (a linear map is fitted per feature on the matched rows, R² must be > 0.999).
3. Keep the transactions that are **not** in the competition files: the unlabelled transactions removed from `test.csv` (and any others), converted to the competition's feature scale.

If the match fails, the rest of the notebook is skipped and the summary says so.

In [10]:
# ===== 9. Public Elliptic features (labels file deliberately never read) =====
import glob as _glob, zipfile as _zipfile
NAME = 'elliptic_txs_features.csv'
# Optional: if the search below fails, paste the full path to the file (or to the downloaded .zip) here
PUB_PATH_MANUAL = ''

def find_public():
    if PUB_PATH_MANUAL:
        return PUB_PATH_MANUAL
    roots = [DATA_DIR, '.', '..', '../..', os.path.expanduser('~/Downloads'), '/kaggle/input']
    for r in roots:                                    # 1) the csv itself, anywhere under these folders
        hits = _glob.glob(os.path.join(r, '**', NAME), recursive=True)
        if hits:
            return hits[0]
    for r in roots:                                    # 2) a zip that contains it (e.g. Kaggle's archive.zip)
        for z in _glob.glob(os.path.join(r, '**', '*.zip'), recursive=True):
            try:
                with _zipfile.ZipFile(z) as zf:
                    if any(n.endswith(NAME) for n in zf.namelist()):
                        return z
            except Exception:
                pass
    return None

def read_public(path):
    if path.endswith('.zip'):
        with _zipfile.ZipFile(path) as zf:
            inner = [n for n in zf.namelist() if n.endswith(NAME)][0]
            with zf.open(inner) as fh:
                return pd.read_csv(fh, header=None)
    return pd.read_csv(path, header=None)

PUB_PATH = find_public()
if PUB_PATH is None:
    # Not found locally: download the public Elliptic dataset with kagglehub (pip install kagglehub)
    try:
        import kagglehub
        path = kagglehub.dataset_download("ellipticco/elliptic-data-set")
        print("Path to dataset files:", path)
        hits = _glob.glob(os.path.join(path, '**', NAME), recursive=True)
        PUB_PATH = hits[0] if hits else None
    except Exception as e:
        print('kagglehub download did not work:', repr(e))
assert PUB_PATH, (f'{NAME} not found in {DATA_DIR}, the notebook folder, the two folders above it, or ~/Downloads '
                  '(also searched inside .zip files), and the kagglehub download failed. Run  pip install kagglehub  or download the Elliptic Data Set from Kaggle '
                  '(ellipticco/elliptic-data-set) or set PUB_PATH_MANUAL to the file path.')
pub = read_public(PUB_PATH)
assert pub.shape[1] - 2 == len(FEAT), f'public file has {pub.shape[1] - 2} features, competition has {len(FEAT)}'
pub.columns = ['txId', 'time_step'] + FEAT
pub[FEAT] = pub[FEAT].astype(np.float32)
print(f'Public features: {pub.shape} from {PUB_PATH}')

comp = pd.concat([train[['txId', 'time_step'] + FEAT], test[['txId', 'time_step'] + FEAT]], ignore_index=True)
id_match = float(comp['txId'].isin(pub['txId']).mean())
print(f'Competition transactions found in the public file by txId: {id_match:.1%}')
MATCHED = id_match > 0.95
coef = {}
if MATCHED:
    mm = comp.merge(pub, on='txId', suffixes=('', '_pub'))
    step_ok = float((mm['time_step'] == mm['time_step_pub']).mean())
    r2s = []
    for c in FEAT:
        xp, xc = mm[c + '_pub'].values.astype(np.float64), mm[c].values.astype(np.float64)
        (a, b), *_ = np.linalg.lstsq(np.c_[xp, np.ones_like(xp)], xc, rcond=None)
        ss = ((xc - xc.mean()) ** 2).sum()
        r2s.append(1 - ((xc - (a * xp + b)) ** 2).sum() / ss if ss > 0 else 1.0)
        coef[c] = (a, b)
    print(f'Time steps agree: {step_ok:.1%} | feature rescaling fit R^2: min {min(r2s):.5f}, median {np.median(r2s):.5f}')
    MATCHED = step_ok > 0.99 and float(np.median(r2s)) > 0.999
    del mm
if MATCHED:
    extra = pub[~pub['txId'].isin(comp['txId'])].copy()
    for c in FEAT:
        a, b = coef[c]
        extra[c] = (a * extra[c].astype(np.float64) + b).astype(np.float32)
    t_min = test['time_step'].min()
    print(f'Extra transactions (in the public file, not in the competition files): {len(extra):,} '
          f'({(extra["time_step"] >= t_min).sum():,} in the test period)')
    all_ids = set(comp['txId']) | set(extra['txId'])
    e_cov = float((edges.iloc[:, 0].isin(all_ids) & edges.iloc[:, 1].isin(all_ids)).mean())
    print(f'Edges with both ends now available: {e_cov:.1%} (was {1 - REPORT["edges_missing_endpoint"] / len(edges):.1%})')
    REPORT['public_match_rate'], REPORT['public_extra_rows'], REPORT['edge_coverage_full'] = id_match, len(extra), e_cov
else:
    print('Could not match the public file to the competition data - the rest of this notebook will not write files.')
del comp

Public features: (203769, 167) from ./data/elliptic_bitcoin_dataset/elliptic_txs_features.csv
Competition transactions found in the public file by txId: 100.0%
Time steps agree: 100.0% | feature rescaling fit R^2: min 1.00000, median 1.00000
Extra transactions (in the public file, not in the competition files): 46,668 (46,668 in the test period)
Edges with both ends now available: 100.0% (was 75.1%)


## 10. Full-graph features (every transaction, every period)
Built exactly like v1's graph features, but now valid: train and test transactions both see **all** their neighbours. Also: per-step ranks over all transactions, and 2-hop / spread / difference features over the full graph. The drift check at the end compares labelled training vs test transactions on the full-graph structural features (0.5 = identical; v1's broken version was ≈ 0.95–1.0).

In [11]:
# ===== 10. Full-graph features =====
if MATCHED:
    t0 = time.time()
    extra['is_test'], extra['y'] = 2, np.nan
    cols0 = ['txId', 'time_step', 'is_test', 'y'] + FEAT
    AN = pd.concat([train[cols0], test.assign(y=np.nan)[cols0], extra[cols0]], ignore_index=True)
    assert AN['txId'].is_unique
    gF, _ = graph_features(AN)
    gF.columns = ['F_' + c for c in gF.columns]
    rF = AN.groupby('time_step')[LOCAL].rank(pct=True).astype(np.float32)
    rF.columns = ['Frank_' + c for c in LOCAL]

    pos_an = pd.Series(np.arange(len(AN)), index=AN['txId'].values)
    s_, d_ = edges.iloc[:, 0].map(pos_an), edges.iloc[:, 1].map(pos_an)
    k_ = (s_.notna() & d_.notna()).values
    s_, d_ = s_[k_].astype(np.int64).values, d_[k_].astype(np.int64).values
    A_ = sp.csr_matrix((np.ones(len(s_), dtype=np.float32), (s_, d_)), shape=(len(AN), len(AN)))
    A_.data[:] = 1.0
    AT_ = A_.T.tocsr()
    UF = ((A_ + AT_) > 0).astype(np.float32).tocsr()
    X_ = AN[LOCAL].values.astype(np.float32)
    has_nb_ = np.asarray(UF.sum(axis=1)).ravel() > 0
    m_u_ = mean_over_neighbours(UF, X_)
    eF = pd.concat([
        pd.DataFrame(mean_over_neighbours(two_hop(AT_), X_), columns=['F2_pred2_' + c for c in LOCAL]),
        pd.DataFrame(mean_over_neighbours(two_hop(A_), X_), columns=['F2_succ2_' + c for c in LOCAL]),
        pd.DataFrame(np.sqrt(np.maximum(mean_over_neighbours(UF, X_ ** 2) - m_u_ ** 2, 0)), columns=['F2_std_' + c for c in LOCAL]),
        pd.DataFrame(np.where(has_nb_[:, None], X_ - m_u_, 0), columns=['F2_diff_' + c for c in LOCAL]),
    ], axis=1).astype(np.float32)
    del X_, m_u_
    AN = pd.concat([AN, gF, rF, eF], axis=1)
    # Component size is left out: with every transaction included, each time step forms one large connected
    # group, so component size would effectively tell the model which time step a row comes from (pure drift).
    GRAPH_F = [c for c in gF.columns if c != 'F_g_comp_size']
    RANK_F, EXT_F = list(rF.columns), list(eF.columns)
    GSTRUCT_F = ['F_' + c for c in GSTRUCT if c != 'g_comp_size']
    del gF, rF, eF

    L_rows = pos_an[L['txId'].values].values          # L row i  -> AN row L_rows[i]
    T_rows = pos_an[T['txId'].values].values          # T row j  -> AN row T_rows[j]
    assert (AN['y'].values[L_rows].astype(int) == y).all()
    cmpF = pd.DataFrame({'train (labelled)': AN.loc[L_rows, GSTRUCT_F].mean().values,
                         'test': AN.loc[T_rows, GSTRUCT_F].mean().values}, index=GSTRUCT_F).round(2)
    display(cmpF)
    adv_F = adversarial_auc(AN.loc[L_rows, GSTRUCT_F].reset_index(drop=True), AN.loc[T_rows, GSTRUCT_F].reset_index(drop=True))
    print(f'Drift AUC on full-graph structural features: {adv_F:.3f} (v1 broken graph: {REPORT["adv_auc_graph_v1"]:.3f}, '
          f'v2 labelled-only: {REPORT["adv_auc_graph_v2"]:.3f}; 0.5 = no drift) | {time.time() - t0:.0f}s')
    REPORT['adv_auc_graph_full_public'] = adv_F

,train (labelled),test
F_g_in_deg,1.76,2.02
F_g_out_deg,1.10,1.22
F_g_deg,2.87,3.24
F_g_in_out_ratio,1.86,1.85
F_g_out_2hop,0.93,1.49
F_g_in_2hop,1.52,2.11
F_g_pred_mean_out_deg,10.61,7.10
F_g_succ_mean_in_deg,8.85,16.02


Drift AUC on full-graph structural features: 0.764 (v1 broken graph: 1.000, v2 labelled-only: 0.931; 0.5 = no drift) | 4s


## 11. Final models

**LightGBM settings (all models):** binary objective, learning rate 0.04, 31 leaves, ≥ 30 rows per leaf, 80% row subsampling, 50% column sampling per tree, L2 = 1, deterministic training. Class imbalance (≈ 12% illicit) is not re-weighted: ROC-AUC depends only on ranking, and a class-balanced variant validated worse.

**Window models:** for a window of K steps, each validation fold trains only on the K steps before its validation window; the tree count is the peak of the fold-averaged validation AUC curve. The final model is trained on the last K training steps (steps 36−K to 35) with that tree count + 10%, averaged over 10 random seeds.

| Ingredient | Features | Training window |
|---|---|---|
| `w16`, `w18`, `v4` (=20), `w22`, `w24` | 165 original + labelled-only graph features (feature set C) | last 16 / 18 / 20 / 22 / 24 steps |
| `wall` | feature set C | all 35 steps |
| `fg` | 165 original + full-graph features (+ extended full-graph features if they validate better) | last 20 steps |

**Files:**
- **E1** = equal rank average of `w16, w18, v4, w22, w24` (the **window bag**).
- **E2** = 80% window bag + 20% `fg`.
- **E3** = 80% window bag + 20% `wall`.
- **Top-3** = equal rank average of E1, E2, E3.

Rank averaging converts each model's scores to percentiles (0–1) before averaging, which suits AUC (only the ordering matters) and stops one model dominating through its probability scale.

In [12]:
# ===== 11. Final models =====
assert MATCHED, 'The public Elliptic features file must be matched in section 9 (needed for the fg ingredient).'
t0_all = time.time()

# Design matrices: feature set C (original + labelled-only graph) plus the full-graph columns for fg
FCOLS = GRAPH_F + EXT_F
LX = pd.concat([L[FEAT + GRAPH].reset_index(drop=True), AN.loc[L_rows, FCOLS].reset_index(drop=True)], axis=1)
TX = pd.concat([T[FEAT + GRAPH].reset_index(drop=True), AN.loc[T_rows, FCOLS].reset_index(drop=True)], axis=1)

def cv_model(name, cols, K=20):
    """Time-based validation for a K-step training window; returns the shared best tree count."""
    t0 = time.time()
    fk = recent_folds(K) if K else folds_full
    models, curves = [], []
    for tr, va in fk:
        m = fit_lgb(LX.loc[tr, cols], y[tr], BASE_PARAMS, X_va=LX.loc[va, cols], y_va=y[va])
        models.append(m)
        curves.append(np.asarray(m.evals_result_['valid_0']['auc']))
    n = min(len(c) for c in curves)
    best = int(np.argmax(np.mean([c[:n] for c in curves], axis=0))) + 1       # one tree count for all folds
    preds = [m.predict_proba(LX.loc[va, cols], num_iteration=best)[:, 1] for m, (_, va) in zip(models, fk)]
    aucs = [roc_auc_score(y[va], p) for p, (_, va) in zip(preds, fk)]
    print(f'{name:48s} validation AUC {np.mean(aucs):.4f} | folds {np.round(aucs, 4)} | trees {best} | {time.time() - t0:.0f}s')
    return dict(name=name, mean=float(np.mean(aucs)), aucs=aucs, iters=best, preds=preds, cols=cols, K=K)

def test_pred(m, seeds):
    """Refit on the last K training steps (or all steps) and predict the test set, averaged over seeds."""
    K = m['K']
    tr = np.where((L['time_step'] >= L['time_step'].max() + 1 - K).values)[0] if K else np.arange(len(L))
    n_iter = max(100, int(m['iters'] * 1.1))
    ms = [fit_lgb(LX.loc[tr, m['cols']], y[tr], BASE_PARAMS, n_estimators=n_iter, seed=s) for s in seeds]
    return np.mean([mm.predict_proba(TX[m['cols']])[:, 1] for mm in ms], axis=0)

C_COLS = FEAT + GRAPH
M = {}
M['v4'] = cv_model('window 20 (v4)', C_COLS, K=20)
M['w16'] = cv_model('window 16', C_COLS, K=16)
M['w18'] = cv_model('window 18', C_COLS, K=18)
M['w22'] = cv_model('window 22', C_COLS, K=22)
M['w24'] = cv_model('window 24', C_COLS, K=24)
M['wall'] = cv_model('all time steps', C_COLS, K=None)
fg_runs = [cv_model('full graph (last 20)', FEAT + GRAPH_F, K=20),
           cv_model('full graph + extended (last 20)', FEAT + GRAPH_F + EXT_F, K=20)]
M['fg'] = max(fg_runs, key=lambda r: r['mean'])
print('  -> full-graph ingredient:', M['fg']['name'])

# 10 seeds per ingredient: the original 5 seeds + 5 extra seeds, averaged
SEEDS_A = [42, 7, 2026, 11, 123]
SEEDS_B = [101, 202, 303, 404, 505]
P10 = {}
for k, m in M.items():
    P10[k] = 0.5 * test_pred(m, SEEDS_A) + 0.5 * test_pred(m, SEEDS_B)
    print('test predictions ready:', k)

WIDE = ['w16', 'w18', 'v4', 'w22', 'w24']
E1 = sum(rank01(P10[k]) / 5 for k in WIDE)
E2 = sum(0.8 / 5 * rank01(P10[k]) for k in WIDE) + 0.2 * rank01(P10['fg'])
E3 = sum(0.8 / 5 * rank01(P10[k]) for k in WIDE) + 0.2 * rank01(P10['wall'])
TOP3 = np.mean([rank01(E1), rank01(E2), rank01(E3)], axis=0)

save_submission(E1, 'reproduced_submission_v10_E1.csv')
save_submission(TOP3, 'reproduced_submission_final_top3.csv')

# Validation of the blends on the same folds (for the report)
def blend_cv(parts):
    return float(np.mean([roc_auc_score(y[va], sum(w * rank01(M[k]['preds'][f]) for k, w in parts))
                          for f, (_, va) in enumerate(recent_folds(20))]))
print(f'\nTime-based validation AUC - E1: {blend_cv([(k, 0.2) for k in WIDE]):.4f} | '
      f'E2: {blend_cv([(k, 0.16) for k in WIDE] + [("fg", 0.2)]):.4f} | '
      f'E3: {blend_cv([(k, 0.16) for k in WIDE] + [("wall", 0.2)]):.4f}')
print(f'Section 11 done in {time.time() - t0_all:.0f}s')

window 20 (v4)                                   validation AUC 0.9821 | folds [0.9518 0.996  0.9986] | trees 153 | 37s
window 16                                        validation AUC 0.9784 | folds [0.9399 0.9966 0.9988] | trees 273 | 35s
window 18                                        validation AUC 0.9800 | folds [0.9457 0.9958 0.9985] | trees 218 | 37s
window 22                                        validation AUC 0.9819 | folds [0.9518 0.9954 0.9985] | trees 153 | 38s
window 24                                        validation AUC 0.9818 | folds [0.9518 0.9953 0.9984] | trees 154 | 40s
all time steps                                   validation AUC 0.9815 | folds [0.9518 0.9945 0.9983] | trees 153 | 41s
full graph (last 20)                             validation AUC 0.9780 | folds [0.9401 0.9954 0.9985] | trees 175 | 35s
full graph + extended (last 20)                  validation AUC 0.9784 | folds [0.9404 0.9959 0.9989] | trees 168 | 63s
  -> full-graph ingredient: full graph +

## 12. Reproducibility check
Compares the regenerated files with the files that were submitted (if they are in the project folder). LightGBM is run in deterministic mode with fixed seeds, so the rankings should match (rank correlation ≈ 1.000).

In [13]:
# ===== 12. Reproducibility check =====
from scipy.stats import spearmanr
for orig, new in [('submission_v10_E1.csv', 'reproduced_submission_v10_E1.csv'),
                  ('submission_final_top3.csv', 'reproduced_submission_final_top3.csv')]:
    o = None
    for root in ['.', '..', '../..']:
        hits = glob.glob(os.path.join(root, '**', orig), recursive=True)
        if hits:
            o = pd.read_csv(hits[0])['target'].values
            break
    n = pd.read_csv(os.path.join(OUT_DIR, new))['target'].values
    if o is None:
        print(f'{orig}: original not found in the project folder - skipped')
    else:
        print(f'{new} vs {orig}: rank correlation {spearmanr(o, n)[0]:.6f} | largest difference {np.abs(o - n).max():.2e}')

submission_v10_E1.csv: original not found in the project folder - skipped
submission_final_top3.csv: original not found in the project folder - skipped


## 13. Summary of the approach (see the report for the full discussion)
- **Key fix:** graph features built on a labelled-only graph so train and test transactions are described consistently (public AUC 0.941 → 0.953).
- **Recency:** training on the most recent 20 steps beats training on all 35 (0.953 → 0.955): illicit behaviour drifts over time.
- **Averaging:** bagging over several windows and seeds reduces variance (0.955 → 0.9558), and a rank average of three closely related variants gives the best public score (0.95585).
- **Rejected by validation and/or the public leaderboard:** Random Forest / Extra Trees blends, pseudo-labelling and self-training, importance weighting, bigger or smaller models, per-step features, DART, HistGradientBoosting, two-stage graph stacking, row-order features.